# YOLOv8 Scene Feature Extraction — Colab Driver

Runs `scripts/extract_yolo_features.py` against all 626 labeled clips on Colab's GPU.
Expected runtime on L4: ~18–25 min end-to-end; T4 free tier: ~45 min.

### Before running
1. **Runtime → Change runtime type → GPU** (L4 Pro / T4 free).
2. Add the following **Colab Secrets** (🔑 icon in left sidebar, toggle each to ON for this notebook):
   - `R2_ACCOUNT_ID`
   - `R2_ACCESS_KEY_ID`
   - `R2_SECRET_ACCESS_KEY`
   - `R2_BUCKET_NAME`
3. Run every cell top-to-bottom.

No database access is required — the notebook reads `data/clip_manifest.json` (committed in the repo) for the list of clip IDs and R2 keys, then pulls videos directly from R2.

The notebook clones the repo, installs deps, runs feature extraction, and zips the `data/processed/*_yolo.npz` outputs for download.

## 1 · Environment + secrets

In [ ]:
!nvidia-smi || echo 'No GPU attached — switch runtime to GPU.'

from google.colab import userdata
import os
for k in ["R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET_NAME"]:
    try:
        os.environ[k] = userdata.get(k)
    except Exception as exc:
        print(f'⚠️  missing secret: {k}  ({exc})')

## 2 · Clone repo + install deps

In [ ]:
REPO_URL = 'https://github.com/yash-b18/Personal-Dashcam.git'
BRANCH = 'feature/ui-improvements'

!rm -rf /content/dashcam && git clone --depth 1 --branch $BRANCH $REPO_URL /content/dashcam
%cd /content/dashcam

!pip install -q ultralytics opencv-python-headless boto3 tqdm decord

## 3 · Download YOLOv8m weights

Weights are `models/yolov8m.pt` in the repo but not committed (large). Pull from ultralytics CDN on first run.

In [ ]:
from pathlib import Path
Path('models').mkdir(exist_ok=True)
if not Path('models/yolov8m.pt').exists():
    !wget -q https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8m.pt -O models/yolov8m.pt
print('yolov8m.pt size:', Path('models/yolov8m.pt').stat().st_size / 1e6, 'MB')

## 4 · Run extraction on GPU

Pulls each clip from R2 → samples frames at 5 fps → runs YOLOv8m batched → writes per-clip `*_yolo.npz` into `data/processed/`.

In [ ]:
!python scripts/extract_yolo_features.py --manifest data/clip_manifest.json --device cuda

## 5 · Zip outputs and download

Compact: 626 × ~200 bytes of npz ≈ <1 MB total, trivial to download.

In [ ]:
import shutil
from pathlib import Path

npz = sorted(Path('data/processed').glob('*_yolo.npz'))
print(f'Produced {len(npz)} YOLO feature files')

archive = shutil.make_archive('/content/yolo_features', 'zip', 'data/processed',
                              base_dir=None)
print('archive:', archive)

from google.colab import files
files.download('/content/yolo_features.zip')

## 6 · Locally, unzip into `data/processed/`

```bash
unzip ~/Downloads/yolo_features.zip -d data/processed/
ls data/processed/*_yolo.npz | wc -l   # should print 626
```

Next step: `scripts/train_all.py` (coming next) trains classical-19, classical+YOLO (35-dim), and LSTM+YOLO on the locked split in `data/splits.json`, and writes a side-by-side comparison to `data/outputs/model_comparison.json`.